# LLM-Assisted Semiconductor MAP TXT Converter
Upload a vendor MAP file → AI proposes transformation rules → Deterministic conversion to ASEKR standard format

In [ ]:
# Cell 1: Setup & Dependencies
!pip install -q huggingface_hub pydantic

import json, re, os, math
from typing import Optional, List, Dict, Any
from getpass import getpass
from pathlib import Path
import chardet
from pydantic import BaseModel, Field
from huggingface_hub import InferenceClient
from IPython.display import display, HTML

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

HF_TOKEN = getpass("Enter your Hugging Face API token: ")

In [ ]:
# Cell 2: Configuration & Data Models
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
CONFIDENCE_THRESHOLD = 0.7
SAMPLE_LINE_COUNT = 15
OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

class FieldExtraction(BaseModel):
    source: str
    pattern: str
    example_value: str

class BinMapping(BaseModel):
    source_code: str
    output_char: str
    meaning: str
    is_pass: bool

class EvidenceSnippet(BaseModel):
    line_number: int
    content: str
    relevance: str

class MapParsing(BaseModel):
    type: str
    grid_start_line: Optional[int] = None
    grid_end_line: Optional[int] = None
    coordinate_pattern: Optional[str] = None
    null_character: str = "."

class RuleProposal(BaseModel):
    detected_format: Dict[str, Any]
    header_rules: Dict[str, Any]
    metadata_extraction: Dict[str, Any]
    map_parsing: MapParsing
    bin_mapping: List[BinMapping]
    confidence: Dict[str, float]
    candidates: Optional[Dict[str, Any]] = None
    evidence: List[EvidenceSnippet]

class HeuristicAnalysis(BaseModel):
    encoding: str
    total_lines: int
    format_type: str
    likely_delimiter: Optional[str] = None
    header_end_line: int
    sample_top: List[str]
    sample_middle: List[str]
    sample_bottom: List[str]

class ASEKROutput(BaseModel):
    version: str = "001"
    customer: str = "[UNKNOWN - please fill in]"
    supplier: str = "N/A"
    wafer_id: str = "[UNKNOWN - please fill in]"
    wafer_lot: str = "[UNKNOWN - please fill in]"
    wafer_no: str = "[UNKNOWN - please fill in]"
    row_y: int = 0
    column_x: int = 0
    notch: str = ""
    good_bin_code: str = "1"
    good_bin_count: int = 0
    total_good_qty: int = 0
    map_grid: List[str] = []

class ValidationReport(BaseModel):
    input_file: str
    fields_found: List[str] = []
    fields_missing: List[str] = []
    missing_field_rate: float = 0.0
    total_input_dies: int = 0
    total_output_dies: int = 0
    parse_errors: List[str] = []
    parse_error_count: int = 0
    good_die_count: int = 0
    confidence_summary: Dict[str, float] = {}

print("Configuration loaded.")

In [ ]:
# Cell 3: Encoding Detection & File Reader
def detect_encoding(raw_bytes: bytes) -> str:
    result = chardet.detect(raw_bytes)
    if result and result.get('encoding'):
        enc = result['encoding'].lower()
        if enc in ('ascii', 'utf-8', 'utf8'):
            return 'utf-8'
        return enc
    return 'utf-8'

def read_file(file_path: str) -> tuple:
    with open(file_path, 'rb') as f:
        raw = f.read()
    encoding = detect_encoding(raw)
    for enc in [encoding, 'utf-8', 'cp949', 'euc-kr', 'latin-1']:
        try:
            return raw.decode(enc), enc
        except (UnicodeDecodeError, LookupError):
            continue
    return raw.decode('latin-1'), 'latin-1'

print("File reader ready.")

In [ ]:
# Cell 4: Heuristic Analyzer
def analyze_heuristics(content: str, encoding: str, n: int = SAMPLE_LINE_COUNT) -> HeuristicAnalysis:
    lines = content.splitlines()
    total = len(lines)
    top = lines[:n]
    mid_start = max(0, total // 2 - n // 2)
    middle = lines[mid_start:mid_start + n]
    bottom = lines[max(0, total - n):]

    format_type = "unknown"
    header_end = 0
    delimiter = None

    if any('WAFER_MAP' in l for l in lines[:5]):
        format_type = "structured_kv"
        for i, l in enumerate(lines):
            if l.strip().startswith('MAP = {') or l.strip() == 'MAP = {':
                header_end = i + 1
                break
    elif any('[BOF]' in l for l in lines[:3]):
        format_type = "structured_kv"
        for i, l in enumerate(lines):
            if '[SOFT BIN MAP]' in l:
                header_end = i + 1
                break
    elif any(re.match(r'^X=\s*\d+\s+Y=\s*\d+\s+B=\s*\d+', l) for l in lines[20:60]):
        format_type = "coordinate_xy_b"
        for i, l in enumerate(lines):
            if re.match(r'^X=\s*\d+\s+Y=\s*\d+\s+B=\s*\d+', l):
                header_end = i
                break
    else:
        non_empty = [l for l in lines if l.strip()]
        if non_empty:
            grid_like = 0
            for l in non_empty[:30]:
                s = l.strip()
                if s and len(s) > 10 and all(c in '.0123456789XYZGOCJxyzgocj ' for c in s):
                    grid_like += 1
            if grid_like > len(non_empty[:30]) * 0.5:
                format_type = "grid_map"
                for i, l in enumerate(lines):
                    s = l.strip()
                    if s and len(s) > 10 and all(c in '.0123456789XYZGOCJxyzgocj ' for c in s):
                        header_end = i
                        break

    return HeuristicAnalysis(
        encoding=encoding,
        total_lines=total,
        format_type=format_type,
        likely_delimiter=delimiter,
        header_end_line=header_end,
        sample_top=top,
        sample_middle=middle,
        sample_bottom=bottom
    )

print("Heuristic analyzer ready.")

In [ ]:
# Cell 5: LLM Provider Abstraction
class LLMProvider:
    def generate_rules(self, prompt: str, schema: dict) -> dict:
        raise NotImplementedError

class HuggingFaceProvider(LLMProvider):
    def __init__(self, model_id: str, token: str):
        self.client = InferenceClient(api_key=token)
        self.model_id = model_id

    def generate_rules(self, prompt: str, schema: dict) -> dict:
        response = self.client.chat_completion(
            model=self.model_id,
            messages=[
                {"role": "system", "content": "You are a JSON-only responder. Output raw JSON with no markdown formatting."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=4096,
            temperature=0.1,
        )
        text = response.choices[0].message.content.strip()
        if text.startswith("```"):
            text = re.sub(r'^```(?:json)?\s*', '', text)
            text = re.sub(r'\s*```$', '', text)
        return json.loads(text)

llm_provider = HuggingFaceProvider(LLM_MODEL, HF_TOKEN)
print(f"LLM Provider ready: {LLM_MODEL}")

In [ ]:
# Cell 6: Rule Generator
RULE_SCHEMA_HINT = """{
  "detected_format": {"encoding": str, "delimiter": str|null, "fixed_width": bool, "format_type": str},
  "header_rules": {"lines_to_skip": int, "detection_reason": str},
  "metadata_extraction": {
    "lot_id": {"source": str, "pattern": str, "example_value": str},
    "wafer_no": {"source": str, "pattern": str, "example_value": str},
    "customer": {"source": str, "pattern": str, "example_value": str} | null,
    "supplier": {"source": str, "pattern": str, "example_value": str} | null,
    "wafer_id": {"source": str, "pattern": str, "example_value": str} | null,
    "notch": {"source": str, "pattern": str, "example_value": str} | null,
    "row_count": {"source": str, "pattern": str, "example_value": str} | null,
    "column_count": {"source": str, "pattern": str, "example_value": str} | null,
    "good_bin_code": {"source": str, "pattern": str, "example_value": str} | null,
    "good_die_count": {"source": str, "pattern": str, "example_value": str} | null
  },
  "map_parsing": {"type": "grid_character"|"coordinate_records"|"structured_grid", "grid_start_line": int|null, "grid_end_line": int|null, "coordinate_pattern": str|null, "null_character": str},
  "bin_mapping": [{"source_code": str, "output_char": str, "meaning": str, "is_pass": bool}],
  "confidence": {"field_name": float_0_to_1},
  "candidates": {"field_name": [{"source": str, "pattern": str, "example_value": str}]} | null,
  "evidence": [{"line_number": int, "content": str, "relevance": str}]
}"""

def build_prompt(analysis: HeuristicAnalysis, filename: str) -> str:
    sampled = "--- TOP LINES ---\n"
    sampled += "\n".join(f"L{i+1}: {l}" for i, l in enumerate(analysis.sample_top))
    sampled += "\n\n--- MIDDLE LINES ---\n"
    mid_offset = max(0, analysis.total_lines // 2 - SAMPLE_LINE_COUNT // 2)
    sampled += "\n".join(f"L{mid_offset+i+1}: {l}" for i, l in enumerate(analysis.sample_middle))
    sampled += "\n\n--- BOTTOM LINES ---\n"
    bot_offset = max(0, analysis.total_lines - SAMPLE_LINE_COUNT)
    sampled += "\n".join(f"L{bot_offset+i+1}: {l}" for i, l in enumerate(analysis.sample_bottom))

    return f"""You are a semiconductor MAP file format analyzer. Analyze the following vendor MAP file and propose transformation rules to convert it to a standard ASEKR format.

FILENAME: {filename}
ENCODING: {analysis.encoding}
TOTAL LINES: {analysis.total_lines}
DETECTED FORMAT TYPE: {analysis.format_type}
DETECTED HEADER END LINE: {analysis.header_end_line}

SAMPLED FILE CONTENT:
{sampled}

INSTRUCTIONS:
1. Identify metadata fields: lot_id, wafer_no, customer, supplier, wafer_id, notch, row_count, column_count, good_bin_code, good_die_count
2. For each field, specify the source location (header line, filename, etc.) and extraction pattern (regex or key name)
3. Identify the wafer map data format and bin code meanings
4. Propose bin_mapping: how each source bin code maps to a single output character
5. For grid maps: identify which characters represent pass (usually '1') and which represent fail
6. For coordinate records (X/Y/B format): identify the coordinate pattern and bin code field
7. Set confidence scores (0.0-1.0) for each field extraction
8. If unsure about a mapping, include alternatives in candidates

The target ASEKR output format has:
- 12-line header (version, Customer, Supplier, Wafer ID, Wafer lot, Wafer No, Row(Y), Column(X), Notch, Good bin code, Good bin count, Total good qty)
- 38 blank padding lines
- Wafer map grid (single characters per die: '.' for empty, '1' for pass, other chars for fail bins)

Output ONLY valid JSON matching this schema:
{RULE_SCHEMA_HINT}

IMPORTANT: Output raw JSON only. No markdown, no code blocks, no explanation."""

def generate_rules(analysis: HeuristicAnalysis, filename: str, max_retries: int = 2) -> RuleProposal:
    prompt = build_prompt(analysis, filename)
    schema = RuleProposal.model_json_schema()
    for attempt in range(max_retries + 1):
        try:
            raw = llm_provider.generate_rules(prompt, schema)
            if 'map_parsing' in raw and isinstance(raw['map_parsing'], dict):
                raw['map_parsing'].setdefault('null_character', '.')
            if 'candidates' not in raw:
                raw['candidates'] = None
            if 'evidence' not in raw:
                raw['evidence'] = []
            if 'bin_mapping' not in raw:
                raw['bin_mapping'] = []
            return RuleProposal.model_validate(raw)
        except Exception as e:
            if attempt == max_retries:
                raise RuntimeError(f"LLM failed after {max_retries+1} attempts: {e}")
            print(f"Retry {attempt+1}/{max_retries}: {e}")

print("Rule generator ready.")

In [ ]:
# Cell 7: Rule Applier
def extract_metadata(content: str, filename: str, rules: RuleProposal) -> ASEKROutput:
    lines = content.splitlines()
    output = ASEKROutput()
    meta = rules.metadata_extraction

    def try_extract(field_def, fallback=None):
        if not field_def or not isinstance(field_def, dict):
            return fallback
        source = field_def.get('source') or ''
        pattern = field_def.get('pattern') or ''
        example = field_def.get('example_value') or fallback
        if 'filename' in source.lower():
            if pattern:
                try:
                    m = re.search(pattern, filename)
                    if m:
                        return m.group(1) if m.groups() else m.group(0)
                except re.error:
                    pass
            return example
        if 'header' in source.lower() or 'line' in source.lower():
            line_match = re.search(r'(\d+)', source)
            if line_match:
                line_idx = int(line_match.group(1)) - 1
                if 0 <= line_idx < len(lines):
                    line = lines[line_idx]
                    if pattern:
                        try:
                            m = re.search(pattern, line)
                            if m:
                                return m.group(1) if m.groups() else m.group(0)
                        except re.error:
                            pass
            for line in lines[:50]:
                if pattern:
                    try:
                        m = re.search(pattern, line)
                        if m:
                            return m.group(1) if m.groups() else m.group(0)
                    except re.error:
                        pass
        return example or fallback

    output.wafer_lot = try_extract(meta.get('lot_id'), '') or ''
    output.wafer_no = try_extract(meta.get('wafer_no'), '') or ''
    output.customer = try_extract(meta.get('customer'), '[UNKNOWN - please fill in]') or '[UNKNOWN - please fill in]'
    output.supplier = try_extract(meta.get('supplier'), 'N/A') or 'N/A'
    output.wafer_id = try_extract(meta.get('wafer_id'), '') or ''
    output.notch = try_extract(meta.get('notch'), '') or ''
    output.good_bin_code = try_extract(meta.get('good_bin_code'), '1') or '1'

    row_str = try_extract(meta.get('row_count'), '')
    col_str = try_extract(meta.get('column_count'), '')
    if row_str:
        try: output.row_y = int(re.search(r'\d+', str(row_str)).group())
        except: pass
    if col_str:
        try: output.column_x = int(re.search(r'\d+', str(col_str)).group())
        except: pass

    good_count_str = try_extract(meta.get('good_die_count'), '')
    if good_count_str:
        try: output.good_bin_count = int(re.search(r'\d+', str(good_count_str).replace(',', '')).group())
        except: pass

    if not output.wafer_lot or not output.wafer_id:
        base = Path(filename).stem
        parts = re.match(r'^([A-Za-z0-9]+)[_.-](?:W?(\d+)|(\d+-\d+))', base)
        if parts:
            if not output.wafer_lot:
                output.wafer_lot = parts.group(1)
            if not output.wafer_no:
                output.wafer_no = parts.group(2) or parts.group(3) or ''
            if not output.wafer_id:
                output.wafer_id = base

    return output

def build_grid_from_grid_map(content: str, rules: RuleProposal) -> list:
    lines = content.splitlines()
    mp = rules.map_parsing
    start = mp.grid_start_line or 0
    end = mp.grid_end_line or len(lines)
    null_char = mp.null_character or '.'
    bin_map = {bm.source_code: bm.output_char for bm in rules.bin_mapping}
    grid = []
    for line in lines[start:end]:
        if not line.strip():
            continue
        stripped = line.rstrip()
        is_metadata = False
        for kw in ['Device:', 'Lot ', 'Slot ', 'Wafer ', 'Flat ', 'Total ', 'Pass ', 'Fail ', 'Yield', 'End of']:
            if kw in stripped:
                is_metadata = True
                break
        if is_metadata:
            continue
        row = ''
        for ch in stripped:
            if ch in bin_map:
                row += bin_map[ch]
            else:
                row += ch
        grid.append(row)
    return grid

def build_grid_from_coordinates(content: str, rules: RuleProposal) -> list:
    lines = content.splitlines()
    pattern = rules.map_parsing.coordinate_pattern or r'X=\s*(\d+)\s+Y=\s*(\d+)\s+B=\s*(\d+)'
    null_char = rules.map_parsing.null_character or '.'
    bin_map = {bm.source_code: bm.output_char for bm in rules.bin_mapping}
    records = []
    for line in lines:
        m = re.match(pattern, line.strip())
        if m:
            x, y, b = int(m.group(1)), int(m.group(2)), m.group(3)
            records.append((x, y, b))
    if not records:
        return []
    min_x = min(r[0] for r in records)
    max_x = max(r[0] for r in records)
    min_y = min(r[1] for r in records)
    max_y = max(r[1] for r in records)
    cols = max_x - min_x + 1
    rows = max_y - min_y + 1
    grid = [[null_char] * cols for _ in range(rows)]
    for x, y, b in records:
        b_padded = b.zfill(4)
        char = bin_map.get(b_padded, bin_map.get(b, bin_map.get(str(int(b)), 'X')))
        grid[y - min_y][x - min_x] = char
    return [''.join(row) for row in grid]

def build_grid_from_structured(content: str, rules: RuleProposal) -> list:
    lines = content.splitlines()
    null_char = rules.map_parsing.null_character or '.'
    bin_map = {bm.source_code: bm.output_char for bm in rules.bin_mapping}
    grid = []
    in_grid = False
    is_bof_format = any('[BOF]' in l for l in lines[:3])
    col_count = 0
    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.startswith('MAP = {') or stripped == 'MAP = {':
            in_grid = True
            continue
        if '[SOFT BIN MAP]' in stripped:
            in_grid = True
            continue
        if in_grid:
            if stripped in ('}', '[EXTENSION]', '[EOF]'):
                break
            if not stripped:
                continue
            if is_bof_format:
                if re.match(r'^\s+\d+\s+\d+\s+\d+', line) and not re.match(r'^\d{3}\s', stripped):
                    continue
                if re.match(r'^\d{3}', stripped):
                    row_data = line[3:] if len(line) > 3 else ''
                    col_width = 4
                    row = ''
                    for ci in range(0, len(row_data), col_width):
                        cell = row_data[ci:ci+col_width].strip()
                        if not cell:
                            row += null_char
                        elif cell in bin_map:
                            row += bin_map[cell]
                        elif cell == '1':
                            row += bin_map.get('1', '1')
                        else:
                            row += 'X'
                    if col_count == 0:
                        col_count = len(row)
                    grid.append(row)
            else:
                row = ''
                for ch in stripped:
                    if ch in bin_map:
                        row += bin_map[ch]
                    else:
                        row += ch
                grid.append(row)
    return grid

def apply_rules(content: str, filename: str, rules: RuleProposal) -> ASEKROutput:
    output = extract_metadata(content, filename, rules)
    map_type = rules.map_parsing.type
    if map_type == 'coordinate_records':
        grid = build_grid_from_coordinates(content, rules)
    elif map_type == 'structured_grid':
        grid = build_grid_from_structured(content, rules)
    else:
        grid = build_grid_from_grid_map(content, rules)
    output.map_grid = grid
    if grid:
        if output.row_y == 0:
            output.row_y = len(grid)
        if output.column_x == 0:
            output.column_x = max(len(row) for row in grid)
    pass_chars = set()
    for bm in rules.bin_mapping:
        if bm.is_pass:
            pass_chars.add(bm.output_char)
    if not pass_chars:
        pass_chars = {'1'}
    good_count = sum(row.count(c) for row in grid for c in pass_chars)
    if output.good_bin_count == 0:
        output.good_bin_count = good_count
    output.total_good_qty = output.good_bin_count
    return output

print("Rule applier ready.")

In [ ]:
# Cell 8: ASEKR Output Generator
def generate_asekr_output(output: ASEKROutput) -> str:
    header_lines = [
        f"ASEKR standard map version : {output.version}",
        f"Customer : {output.customer}",
        f"Supplier : {output.supplier}",
        f"Wafer ID : {output.wafer_id}",
        f"Wafer lot : {output.wafer_lot}",
        f"Wafer No : {output.wafer_no}",
        f"Row(Y) : {output.row_y}",
        f"Column(X) : {output.column_x}",
        f"Notch : {output.notch}",
        f"Good bin code : {output.good_bin_code}",
        f"Good bin count : {output.good_bin_count}",
        f"Total good qty : {output.total_good_qty}",
    ]
    padding = [''] * 38
    all_lines = header_lines + padding + output.map_grid
    return '\n'.join(all_lines) + '\n'

def save_asekr_output(output: ASEKROutput, filename: str) -> str:
    content = generate_asekr_output(output)
    stem = Path(filename).stem
    out_path = os.path.join(OUTPUT_DIR, f"{stem}_converted.txt")
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(content)
    return out_path

print("ASEKR output generator ready.")

In [ ]:
# Cell 9: Validator
def validate_output(output: ASEKROutput, filename: str, rules: RuleProposal) -> ValidationReport:
    required_fields = ['customer', 'wafer_id', 'wafer_lot', 'wafer_no', 'row_y', 'column_x', 'good_bin_code', 'good_bin_count']
    found = []
    missing = []
    errors = []
    for field in required_fields:
        val = getattr(output, field, None)
        if val and str(val) not in ('0', '', '[UNKNOWN - please fill in]', 'N/A'):
            found.append(field)
        else:
            missing.append(field)
    total_dies = sum(1 for row in output.map_grid for c in row if c != '.')
    pass_chars = set()
    for bm in rules.bin_mapping:
        if bm.is_pass:
            pass_chars.add(bm.output_char)
    if not pass_chars:
        pass_chars = {'1'}
    good_dies = sum(row.count(c) for row in output.map_grid for c in pass_chars)
    if output.row_y != len(output.map_grid):
        errors.append(f"Row count mismatch: header={output.row_y}, actual={len(output.map_grid)}")
    if output.map_grid:
        max_col = max(len(row) for row in output.map_grid)
        if output.column_x != max_col:
            errors.append(f"Column count mismatch: header={output.column_x}, actual={max_col}")
    return ValidationReport(
        input_file=filename,
        fields_found=found,
        fields_missing=missing,
        missing_field_rate=len(missing) / len(required_fields) if required_fields else 0,
        total_input_dies=total_dies,
        total_output_dies=total_dies,
        parse_errors=errors,
        parse_error_count=len(errors),
        good_die_count=good_dies,
        confidence_summary=rules.confidence
    )

print("Validator ready.")

In [ ]:
# Cell 10: Main Workflow
def display_rule_json(rules: RuleProposal):
    rule_dict = rules.model_dump()
    json_str = json.dumps(rule_dict, indent=2, ensure_ascii=False)
    html = f'<details><summary><b>Rule Proposal JSON</b> (click to expand)</summary><pre style="background:#f4f4f4;padding:10px;overflow-x:auto;max-height:500px">{json_str}</pre></details>'
    display(HTML(html))

def display_preview(output: ASEKROutput, rules: RuleProposal):
    conf = rules.confidence
    rows = [
        ('Customer', output.customer, conf.get('customer', '-')),
        ('Supplier', output.supplier, conf.get('supplier', '-')),
        ('Wafer ID', output.wafer_id, conf.get('wafer_id', '-')),
        ('Wafer Lot', output.wafer_lot, conf.get('lot_id', '-')),
        ('Wafer No', output.wafer_no, conf.get('wafer_no', '-')),
        ('Row(Y)', str(output.row_y), conf.get('row_count', '-')),
        ('Column(X)', str(output.column_x), conf.get('column_count', '-')),
        ('Notch', output.notch, conf.get('notch', '-')),
        ('Good Bin Code', output.good_bin_code, conf.get('good_bin_code', '-')),
        ('Good Bin Count', str(output.good_bin_count), '-'),
        ('Total Good Qty', str(output.total_good_qty), '-'),
    ]
    table = '<table style="border-collapse:collapse"><tr style="background:#333;color:#fff"><th style="padding:6px 12px">Field</th><th style="padding:6px 12px">Value</th><th style="padding:6px 12px">Confidence</th></tr>'
    for field, val, c in rows:
        c_str = f"{c:.2f}" if isinstance(c, float) else str(c)
        color = '#d4edda' if isinstance(c, float) and c >= CONFIDENCE_THRESHOLD else '#fff3cd' if isinstance(c, float) else '#fff'
        table += f'<tr style="background:{color}"><td style="padding:4px 12px;border:1px solid #ddd">{field}</td><td style="padding:4px 12px;border:1px solid #ddd">{val}</td><td style="padding:4px 12px;border:1px solid #ddd">{c_str}</td></tr>'
    table += '</table>'
    display(HTML(f'<h3>Extracted Metadata</h3>{table}'))
    if output.map_grid:
        preview_rows = min(10, len(output.map_grid))
        grid_preview = '\n'.join(output.map_grid[:preview_rows])
        display(HTML(f'<h3>Wafer Map Preview (first {preview_rows} of {len(output.map_grid)} rows)</h3><pre style="background:#f4f4f4;padding:10px;font-size:10px;line-height:1.2">{grid_preview}</pre>'))

def display_validation(report: ValidationReport):
    status = 'PASS' if report.missing_field_rate < 0.3 and report.parse_error_count == 0 else 'WARNING'
    color = '#28a745' if status == 'PASS' else '#ffc107'
    html = f'<h3>Validation Report <span style="color:{color}">[{status}]</span></h3>'
    html += '<ul>'
    html += f'<li>Fields found: {len(report.fields_found)}/{len(report.fields_found)+len(report.fields_missing)} ({", ".join(report.fields_found)})</li>'
    if report.fields_missing:
        html += f'<li style="color:orange">Fields missing: {", ".join(report.fields_missing)}</li>'
    html += f'<li>Total dies in output: {report.total_output_dies}</li>'
    html += f'<li>Good die count: {report.good_die_count}</li>'
    if report.parse_errors:
        html += f'<li style="color:red">Parse errors: {", ".join(report.parse_errors)}</li>'
    html += '</ul>'
    display(HTML(html))

def run_conversion():
    print("Upload a vendor MAP file...")
    if IN_COLAB:
        uploaded = colab_files.upload()
        if not uploaded:
            print("No file uploaded.")
            return
        filename = list(uploaded.keys())[0]
        file_path = f"/content/{filename}"
        with open(file_path, 'wb') as f:
            f.write(uploaded[filename])
    else:
        file_path = input("Enter file path: ").strip()
        filename = os.path.basename(file_path)

    if not os.path.exists(file_path):
        print(f"Error: File not found: {file_path}")
        return

    file_size = os.path.getsize(file_path)
    if file_size == 0:
        print("Error: File is empty.")
        return
    if file_size > 10 * 1024 * 1024:
        print(f"Warning: Large file ({file_size/1024/1024:.1f}MB). Processing may be slow.")

    print(f"\nProcessing: {filename}")
    print("=" * 60)

    print("[1/5] Reading file...")
    content, encoding = read_file(file_path)
    lines = content.splitlines()
    print(f"  Encoding: {encoding}, Lines: {len(lines)}")
    if len(lines) < 2:
        print("Error: File has too few lines to analyze.")
        return

    print("[2/5] Analyzing structure...")
    analysis = analyze_heuristics(content, encoding)
    print(f"  Format: {analysis.format_type}, Header ends: line {analysis.header_end_line}")

    print("[3/5] Generating rules via LLM...")
    try:
        rules = generate_rules(analysis, filename)
        print(f"  Bin mappings: {len(rules.bin_mapping)}, Map type: {rules.map_parsing.type}")
    except RuntimeError as e:
        print(f"  Error: {e}")
        print("  LLM rule generation failed. Check your HF token and try again.")
        return

    print("[4/5] Applying rules...")
    asekr_output = apply_rules(content, filename, rules)
    print(f"  Grid: {asekr_output.row_y} rows x {asekr_output.column_x} cols")

    print("[5/5] Validating...")
    report = validate_output(asekr_output, filename, rules)
    print("=" * 60)

    display_preview(asekr_output, rules)
    display_rule_json(rules)
    display_validation(report)

    out_path = save_asekr_output(asekr_output, filename)
    print(f"\nOutput saved: {out_path}")

    rule_path = os.path.join(OUTPUT_DIR, f"{Path(filename).stem}_rules.json")
    with open(rule_path, 'w') as f:
        json.dump(rules.model_dump(), f, indent=2, ensure_ascii=False)
    print(f"Rules saved: {rule_path}")

    if IN_COLAB:
        colab_files.download(out_path)
        colab_files.download(rule_path)

run_conversion()